In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = SparkSession.builder \
        .appName("Assignment 1") \
        .enableHiveSupport() \
        .getOrCreate()

In [9]:
ads_file_path = "/tmp/spark_datasets/assignment_1/ad_campaigns_data.json"
ads = spark.read.json(ads_file_path)
ads.printSchema()
ads.show()

root
 |-- campaign_country: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- os_type: string (nullable = true)
 |-- place_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+----------------+-----------+--------------------+-----------+--------------------+----------+-------+---------+-------------------+
|campaign_country|campaign_id|       campaign_name|device_type|          event_time|event_type|os_type| place_id|            user_id|
+----------------+-----------+--------------------+-----------+--------------------+----------+-------+---------+-------------------+
|             USA|    ABCDFAE|Food category tar...|      apple|2018-10-12T13:10:...|impression|    ios|CASSBB-11|1264374214654454321|
|             USA|    ABCDFAE|Food category tar...|   MOTOROLA|2018-10-12T13:

In [27]:
# Q1. Analyse data for each campaign_id, date, hour, os_type & value to get all the events with counts

# expected Output:
# {
# "campaign_id": "ABCDFAE",
# "date": "2018-10-12",
# "hour": "13",
# "type": "os_type",
# "value": "android",
# "event": {
# "impression": 2,
# "click": 1,
# "video ad": 1
# }
# }

ads = ads.withColumn('event_time', col('event_time').cast("timestamp")) \
         .withColumn('date', to_date(col('event_time'))) \
         .withColumn('hour', hour(col('event_time'))) \
         .withColumn('type', lit('os_type')) \
         .withColumn('value', col('os_type'))

# ads.select(col('campaign_id'),col('date'),col('hour'),col('type'),col('value')).show()
ads.show()
ads.printSchema()
# ads.groupBy(col('event_type')).agg(
# count('*').alias('total_count')).orderBy(col('event_type')).show()

+----------------+-----------+--------------------+-----------+-------------------+----------+-------+---------+-------------------+----------+-------+-------+----+
|campaign_country|campaign_id|       campaign_name|device_type|         event_time|event_type|os_type| place_id|            user_id|      date|   type|  value|hour|
+----------------+-----------+--------------------+-----------+-------------------+----------+-------+---------+-------------------+----------+-------+-------+----+
|             USA|    ABCDFAE|Food category tar...|      apple|2018-10-12 13:10:05|impression|    ios|CASSBB-11|1264374214654454321|2018-10-12|os_type|    ios|  13|
|             USA|    ABCDFAE|Food category tar...|   MOTOROLA|2018-10-12 13:09:04|impression|android|CADGBD-13|1674374214654454321|2018-10-12|os_type|android|  13|
|             USA|    ABCDFAE|Food category tar...|    SAMSUNG|2018-10-12 13:10:10|  video ad|android|BADGBA-12|   5747421465445443|2018-10-12|os_type|android|  13|
|         

In [28]:
pivoted_ads = ads.groupBy("campaign_id", "date", "hour", "type", "value") \
    .pivot("event_type") \
    .count() \
    .fillna(0)
pivoted_ads.show()

+-----------+----------+----+-------+-------+-----+----------+--------+
|campaign_id|      date|hour|   type|  value|click|impression|video ad|
+-----------+----------+----+-------+-------+-----+----------+--------+
|    ABCDFAE|2018-10-12|  13|os_type|android|    1|         1|       1|
|    ABCDFAE|2018-10-12|  13|os_type|    ios|    0|         1|       0|
+-----------+----------+----+-------+-------+-----+----------+--------+



In [31]:
final_ads = pivoted_ads.withColumn('event', struct(
                                                    col('click'),
                                                    col('impression'),
                                                    col('video ad'),
                                                    )).select(col('campaign_id'),col('date'),col('hour'),col('type'),col('value'),col('event'))
final_ads.show(truncate=False)
final_ads.printSchema()

+-----------+----------+----+-------+-------+---------+
|campaign_id|date      |hour|type   |value  |event    |
+-----------+----------+----+-------+-------+---------+
|ABCDFAE    |2018-10-12|13  |os_type|android|{1, 1, 1}|
|ABCDFAE    |2018-10-12|13  |os_type|ios    |{0, 1, 0}|
+-----------+----------+----+-------+-------+---------+

root
 |-- campaign_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- type: string (nullable = false)
 |-- value: string (nullable = true)
 |-- event: struct (nullable = false)
 |    |-- click: long (nullable = true)
 |    |-- impression: long (nullable = true)
 |    |-- video ad: long (nullable = true)



In [49]:
final_ads.write.json('/tmp/spark_output/assignment_1/assignment_1_q1.json')

In [50]:
ads_output_file_path = "/tmp/spark_output/assignment_1/assignment_1_q1.json"
ads_output = spark.read.json(ads_output_file_path)
ads_output.printSchema()
ads_output.show()

root
 |-- campaign_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- event: struct (nullable = true)
 |    |-- click: long (nullable = true)
 |    |-- impression: long (nullable = true)
 |    |-- video ad: long (nullable = true)
 |-- hour: long (nullable = true)
 |-- type: string (nullable = true)
 |-- value: string (nullable = true)

+-----------+----------+---------+----+-------+-------+
|campaign_id|      date|    event|hour|   type|  value|
+-----------+----------+---------+----+-------+-------+
|    ABCDFAE|2018-10-12|{1, 1, 1}|  13|os_type|android|
|    ABCDFAE|2018-10-12|{0, 1, 0}|  13|os_type|    ios|
+-----------+----------+---------+----+-------+-------+



In [10]:
store_file_path = "/tmp/spark_datasets/assignment_1/store_data.json"
store = spark.read.json(store_file_path)

store.printSchema()
store.show()

root
 |-- place_ids: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- store_name: string (nullable = true)

+--------------------+-------------+
|           place_ids|   store_name|
+--------------------+-------------+
|[CASSBB-11, CADGB...|     McDonald|
|         [CASSBB-11]|   BurgerKing|
|[BADGBA-13, CASSB...|        Macys|
|         [BADGBA-12]|shoppers stop|
+--------------------+-------------+



In [38]:
# Q2.Analyse data for each campaign_id, date, hour, store_name & value to get all the events with counts
store_explode = store.select(
    col('store_name'),
    explode(col('place_ids')).alias('place_id')
)

store_explode.show()

+-------------+---------+
|   store_name| place_id|
+-------------+---------+
|     McDonald|CASSBB-11|
|     McDonald|CADGBD-13|
|     McDonald|FDBEGD-14|
|   BurgerKing|CASSBB-11|
|        Macys|BADGBA-13|
|        Macys|CASSBB-15|
|        Macys|FDBEGD-15|
|shoppers stop|BADGBA-12|
+-------------+---------+



In [45]:
store_result = ads.alias('ads_data').join(store_explode.alias('ste'),col('ads_data.place_id') == col('ste.place_id'),'inner').drop('ste.place_id')
store_result.show()

+----------------+-----------+--------------------+-----------+-------------------+----------+-------+---------+-------------------+----------+-------+-------+----+-------------+---------+
|campaign_country|campaign_id|       campaign_name|device_type|         event_time|event_type|os_type| place_id|            user_id|      date|   type|  value|hour|   store_name| place_id|
+----------------+-----------+--------------------+-----------+-------------------+----------+-------+---------+-------------------+----------+-------+-------+----+-------------+---------+
|             USA|    ABCDFAE|Food category tar...|      apple|2018-10-12 13:10:05|impression|    ios|CASSBB-11|1264374214654454321|2018-10-12|os_type|    ios|  13|   BurgerKing|CASSBB-11|
|             USA|    ABCDFAE|Food category tar...|      apple|2018-10-12 13:10:05|impression|    ios|CASSBB-11|1264374214654454321|2018-10-12|os_type|    ios|  13|     McDonald|CASSBB-11|
|             USA|    ABCDFAE|Food category tar...|   M

In [46]:
store_result = store_result.withColumn('type', lit('store_name')) \
         .withColumn('value', col('store_name'))

pivoted_store_result = store_result.groupBy("campaign_id", "date", "hour", "type", "value") \
    .pivot("event_type") \
    .count() \
    .fillna(0)
pivoted_store_result.show()

+-----------+----------+----+----------+-------------+-----+----------+--------+
|campaign_id|      date|hour|      type|        value|click|impression|video ad|
+-----------+----------+----+----------+-------------+-----+----------+--------+
|    ABCDFAE|2018-10-12|  13|store_name|shoppers stop|    0|         0|       1|
|    ABCDFAE|2018-10-12|  13|store_name|     McDonald|    1|         2|       0|
|    ABCDFAE|2018-10-12|  13|store_name|   BurgerKing|    1|         1|       0|
+-----------+----------+----+----------+-------------+-----+----------+--------+



In [47]:
final_store_result = pivoted_store_result.withColumn('event', struct(
                                                    col('click'),
                                                    col('impression'),
                                                    col('video ad'),
                                                    )).select(col('campaign_id'),col('date'),col('hour'),col('type'),col('value'),col('event'))
final_store_result.show(truncate=False)
final_store_result.printSchema()

+-----------+----------+----+----------+-------------+---------+
|campaign_id|date      |hour|type      |value        |event    |
+-----------+----------+----+----------+-------------+---------+
|ABCDFAE    |2018-10-12|13  |store_name|shoppers stop|{0, 0, 1}|
|ABCDFAE    |2018-10-12|13  |store_name|McDonald     |{1, 2, 0}|
|ABCDFAE    |2018-10-12|13  |store_name|BurgerKing   |{1, 1, 0}|
+-----------+----------+----+----------+-------------+---------+

root
 |-- campaign_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- type: string (nullable = false)
 |-- value: string (nullable = true)
 |-- event: struct (nullable = false)
 |    |-- click: long (nullable = true)
 |    |-- impression: long (nullable = true)
 |    |-- video ad: long (nullable = true)



In [48]:
final_store_result.write.json('/tmp/spark_output/assignment_1/assignment_1_q2.json')

In [54]:
store_output_file_path = "/tmp/spark_output/assignment_1/assignment_1_q2.json"
store_output = spark.read.json(store_output_file_path)
store_output.printSchema()
store_output.show()

root
 |-- campaign_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- event: struct (nullable = true)
 |    |-- click: long (nullable = true)
 |    |-- impression: long (nullable = true)
 |    |-- video ad: long (nullable = true)
 |-- hour: long (nullable = true)
 |-- type: string (nullable = true)
 |-- value: string (nullable = true)

+-----------+----------+---------+----+----------+-------------+
|campaign_id|      date|    event|hour|      type|        value|
+-----------+----------+---------+----+----------+-------------+
|    ABCDFAE|2018-10-12|{0, 0, 1}|  13|store_name|shoppers stop|
|    ABCDFAE|2018-10-12|{1, 2, 0}|  13|store_name|     McDonald|
|    ABCDFAE|2018-10-12|{1, 1, 0}|  13|store_name|   BurgerKing|
+-----------+----------+---------+----+----------+-------------+



In [55]:
user_file_path = "/tmp/spark_datasets/assignment_1/user_profile_data.json"
user = spark.read.json(user_file_path)

user.printSchema()
user.show()

root
 |-- age_group: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- country: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- user_id: string (nullable = true)

+---------+--------------------+-------+------+-------------------+
|age_group|            category|country|gender|            user_id|
+---------+--------------------+-------+------+-------------------+
|    18-25|  [shopper, student]|    USA|  male|1264374214654454321|
|    25-50|            [parent]|    USA|female|1674374214654454321|
|    25-50|[shopper, parent,...|    USA|  male|   5747421465445443|
|      50+|      [professional]|    USA|  male|1864374214654454132|
|    18-25|  [shopper, student]|    USA|female|  14537421465445443|
|      50+|[shopper, profess...|    USA|female|  25547421465445443|
+---------+--------------------+-------+------+-------------------+



In [58]:
# Q3.Analyse data for each campaign_id, date, hour, gender_type & value to get all the events with counts

user_result = ads.alias('ads_data').join(user.alias('usr'),col('ads_data.user_id') == col('usr.user_id'),'inner').drop('usr.user_id')
user_result.show()

+----------------+-----------+--------------------+-----------+-------------------+----------+-------+---------+-------------------+----------+-------+-------+----+---------+--------------------+-------+------+-------------------+
|campaign_country|campaign_id|       campaign_name|device_type|         event_time|event_type|os_type| place_id|            user_id|      date|   type|  value|hour|age_group|            category|country|gender|            user_id|
+----------------+-----------+--------------------+-----------+-------------------+----------+-------+---------+-------------------+----------+-------+-------+----+---------+--------------------+-------+------+-------------------+
|             USA|    ABCDFAE|Food category tar...|      apple|2018-10-12 13:10:05|impression|    ios|CASSBB-11|1264374214654454321|2018-10-12|os_type|    ios|  13|    18-25|  [shopper, student]|    USA|  male|1264374214654454321|
|             USA|    ABCDFAE|Food category tar...|   MOTOROLA|2018-10-12 13

In [60]:
user_result = user_result.withColumn('type', lit('gender')) \
         .withColumn('value', col('gender'))

pivoted_user_result = user_result.groupBy("campaign_id", "date", "hour", "type", "value") \
    .pivot("event_type") \
    .count() \
    .fillna(0)
pivoted_store_result.show()

+-----------+----------+----+----------+-------------+-----+----------+--------+
|campaign_id|      date|hour|      type|        value|click|impression|video ad|
+-----------+----------+----+----------+-------------+-----+----------+--------+
|    ABCDFAE|2018-10-12|  13|store_name|shoppers stop|    0|         0|       1|
|    ABCDFAE|2018-10-12|  13|store_name|     McDonald|    1|         2|       0|
|    ABCDFAE|2018-10-12|  13|store_name|   BurgerKing|    1|         1|       0|
+-----------+----------+----+----------+-------------+-----+----------+--------+



In [61]:
final_user_result = pivoted_user_result.withColumn('event', struct(
                                                    col('click'),
                                                    col('impression'),
                                                    col('video ad'),
                                                    )).select(col('campaign_id'),col('date'),col('hour'),col('type'),col('value'),col('event'))
final_user_result.show(truncate=False)
final_user_result.printSchema()

+-----------+----------+----+------+------+---------+
|campaign_id|date      |hour|type  |value |event    |
+-----------+----------+----+------+------+---------+
|ABCDFAE    |2018-10-12|13  |gender|female|{0, 1, 0}|
|ABCDFAE    |2018-10-12|13  |gender|male  |{1, 1, 1}|
+-----------+----------+----+------+------+---------+

root
 |-- campaign_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- type: string (nullable = false)
 |-- value: string (nullable = true)
 |-- event: struct (nullable = false)
 |    |-- click: long (nullable = true)
 |    |-- impression: long (nullable = true)
 |    |-- video ad: long (nullable = true)



In [62]:
final_user_result.write.json('/tmp/spark_output/assignment_1/assignment_1_q3.json')

In [63]:
user_output_file_path = "/tmp/spark_output/assignment_1/assignment_1_q3.json"
user_output = spark.read.json(user_output_file_path)
user_output.printSchema()
user_output.show()

root
 |-- campaign_id: string (nullable = true)
 |-- date: string (nullable = true)
 |-- event: struct (nullable = true)
 |    |-- click: long (nullable = true)
 |    |-- impression: long (nullable = true)
 |    |-- video ad: long (nullable = true)
 |-- hour: long (nullable = true)
 |-- type: string (nullable = true)
 |-- value: string (nullable = true)

+-----------+----------+---------+----+------+------+
|campaign_id|      date|    event|hour|  type| value|
+-----------+----------+---------+----+------+------+
|    ABCDFAE|2018-10-12|{0, 1, 0}|  13|gender|female|
|    ABCDFAE|2018-10-12|{1, 1, 1}|  13|gender|  male|
+-----------+----------+---------+----+------+------+



In [70]:
# Define variables
db_name = "assignment1"
hdfs_path = "hdfs:///user/hive/assignment1_warehouse/"
serde_class = "org.apache.hive.hcatalog.data.JsonSerDe"

# Create the database pointing specifically to your HDFS path
spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name} LOCATION '{hdfs_path}'")

# Switch to the new database
spark.sql(f"USE {db_name}")

# Verify it worked
print(spark.sql("SELECT current_database()").collect())

[Row(current_database()='assignment1')]


In [79]:
# 1. Ensure this is a string, not a DataFrame object
table_name = "assignment1_table" 

# 2. Use 'USING JSON' instead of 'ROW FORMAT SERDE'
create_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {table_name} (
    campaign_id STRING,
    date STRING,
    event STRUCT<click: BIGINT, impression: BIGINT, `video ad`: BIGINT>,
    hour INT,
    type STRING,
    value STRING
)
USING JSON
LOCATION '{hdfs_path}'
"""

# 3. Execute
spark.sql(create_table_query)

print(f"Table {table_name} created successfully using native JSON support.")

Table assignment1_table created successfully using native JSON support.


In [81]:
final_ads.printSchema()

root
 |-- campaign_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- type: string (nullable = false)
 |-- value: string (nullable = true)
 |-- event: struct (nullable = false)
 |    |-- click: long (nullable = true)
 |    |-- impression: long (nullable = true)
 |    |-- video ad: long (nullable = true)



In [82]:
correct_order = ["campaign_id", "date", "event", "hour", "type", "value"]

final_ads_reordered = final_ads.select(*correct_order)

final_ads_reordered.write.mode("append").insertInto(table_name)

print("Data successfully loaded!")

Data successfully loaded!


In [83]:
final_user_result_reordered = final_user_result.select(*correct_order)

final_user_result_reordered.write.mode("append").insertInto(table_name)

print("Data successfully loaded!")

Data successfully loaded!


In [84]:
final_store_result_reordered = final_store_result.select(*correct_order)

final_store_result_reordered.write.mode("append").insertInto(table_name)

print("Data successfully loaded!")

Data successfully loaded!


In [85]:
spark.sql("select * from assignment1_table").show()

+-----------+----------+---------+----+----------+-------------+
|campaign_id|      date|    event|hour|      type|        value|
+-----------+----------+---------+----+----------+-------------+
|    ABCDFAE|2018-10-12|{0, 0, 1}|  13|store_name|shoppers stop|
|    ABCDFAE|2018-10-12|{1, 2, 0}|  13|store_name|     McDonald|
|    ABCDFAE|2018-10-12|{1, 1, 0}|  13|store_name|   BurgerKing|
|    ABCDFAE|2018-10-12|{1, 1, 1}|  13|   os_type|      android|
|    ABCDFAE|2018-10-12|{0, 1, 0}|  13|   os_type|          ios|
|    ABCDFAE|2018-10-12|{0, 1, 0}|  13|    gender|       female|
|    ABCDFAE|2018-10-12|{1, 1, 1}|  13|    gender|         male|
+-----------+----------+---------+----+----------+-------------+



In [86]:
spark.stop()